In [17]:
import os
import torch
from collections import defaultdict
import logging
import pickle
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset


/home/mrenaudin/.conda/envs/leaps3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [36]:

ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

Generating validation split: 100%|██████████| 3760/3760 [00:00<00:00, 352462.52 examples/s]


In [26]:
def create_vocab(train_data, vocab_size):
    counter = defaultdict(int)
    for line in train_data:
        for word in line.replace("\n"," <eos>").split():
            counter[word] += 1

    count_pairs = sorted(counter.items(), key=lambda x: (-x[1], x[0]))[:vocab_size]
    words = [w for (w, v) in count_pairs]
    print(len(counter), count_pairs[vocab_size - 1])
    w2idx = dict(zip(words, range(len(words))))
    idx2w = dict(zip(range(len(words)), words))
    return w2idx, idx2w

In [37]:
w2i, i2w = create_vocab(ds["train"]["text"], 50000)

623942 ('Masand', 62)


In [38]:
len(w2i)

50000

In [39]:
def filter_word(word, vocab):
    if word in vocab:
        return word
    else:
        return "<unk>"

In [40]:
def convert_line(line, vocab):
    return [filter_word(word, vocab) for word in line.replace("\n", " <eos>").split()]

In [41]:
output_path = "/scratch2/mrenaudin/colorlessgreenRNNs/new_dataset"

In [42]:
f_train = open(output_path + "/train.txt", 'w')


In [43]:
for line in ds["train"]["text"]:
    f_train.write(" ".join(convert_line(line, w2i)) + "\n")
f_train.close()

Meme implementation qu'avant, juste avec un dataset et un dataloader.
On garde indice pour les phrase et on shuffle ça dans le dataloader

In [44]:
import os
import torch
from collections import defaultdict
import logging

In [45]:
class Dictionary(object):
    def __init__(self, path):
        self.word2idx = {}
        self.idx2word = []
        self.word2freq = defaultdict(int)

        vocab_path = os.path.join(path, 'vocab.txt')
        try:
            vocab = open(vocab_path, encoding="utf8").read()
            self.word2idx = {w: i for i, w in enumerate(vocab.split())}
            self.idx2word = [w for w in vocab.split()]
            self.vocab_file_exists = True
        except FileNotFoundError:
            logging.info("Vocab file not found, creating new vocab file.")
            self.create_vocab(os.path.join(path, 'train.txt'))
            open(vocab_path,"w").write("\n".join([w for w in self.idx2word]))

    def add_word(self, word):
        self.word2freq[word] += 1
        if word not in self.word2idx:
            self.idx2word.append(word)
            self.word2idx[word] = len(self.idx2word) - 1
        #return self.word2idx[word]

    def __len__(self):
        return len(self.idx2word)

    def create_vocab(self, path):
        with open(path, 'r', encoding="utf8") as f:
            for line in f:
                words = line.split()
                for word in words:
                    self.add_word(word)

In [55]:
class Corpus(object):
    def __init__(self, path):
        self.dictionary = Dictionary(path)
        #self.train = tokenize(self.dictionary, os.path.join(path, 'train.txt'))
        self.valid = tokenize(self.dictionary, os.path.join(path, 'valid.txt'), shuffle=False)
        self.test = tokenize(self.dictionary, os.path.join(path, 'test.txt'), shuffle=False)


In [ ]:
corpus.train = tokenize(self.dictionary, os.path.join(path, 'train.txt'), shuffle=True)

In [47]:
import random
def tokenize(dictionary, path, shuffle=False):
    """Tokenizes a text file for training or testing to a sequence of indices format
       We assume that training and test data has <eos> symbols """
    assert os.path.exists(path)
    with open(path, 'r', encoding="utf8") as f:
        lines = f.readlines()

    if shuffle:
        random.shuffle(lines)
        ntokens = 0
        for line in lines:
            words = line.split()
            ntokens += len(words)

    # Tokenize file content
    with open(path, 'r', encoding="utf8") as f:
        ids = torch.LongTensor(ntokens)
        token = 0
        for line in lines:
            words = line.split()
            for word in words:
                if word in dictionary.word2idx:
                    ids[token] = dictionary.word2idx[word]
                else:
                    ids[token] = dictionary.word2idx["<unk>"]
                token += 1

    return ids

In [57]:
import os
import torch
import random

def tokenize(dictionary, path, shuffle=False):
    """Tokenizes a text file to a sequence of indices format.
       Assumes that training and test data have <eos> symbols.
    """
    assert os.path.exists(path)

    # Read all lines
    with open(path, 'r', encoding="utf8") as f:
        lines = f.readlines()

    if shuffle:
        random.shuffle(lines)

    # Count total number of tokens
    ntokens = 0
    for line in lines:
        words = line.split()
        ntokens += len(words)

    # Allocate tensor
    ids = torch.LongTensor(ntokens)

    # Fill tensor
    token = 0
    for line in lines:
        words = line.split()
        for word in words:
            if word in dictionary.word2idx:
                ids[token] = dictionary.word2idx[word]
            else:
                ids[token] = dictionary.word2idx.get("<unk>", 0)
            token += 1

    return ids


In [49]:
def batchify(data, bsz, device):
    #Just add shuffling ici
    # Work out how cleanly we can divide the dataset into bsz parts.
    nbatch = data.size(0) // bsz
    # Trim off any extra elements that wouldn't cleanly fit (remainders).
    data = data.narrow(0, 0, nbatch * bsz)
    # Evenly divide the data across the bsz batches.
    data = data.view(bsz, -1).t().contiguous()
    # if device = 'cuda':
    #     #data = data.cuda()
    data = data.to(device)
    return data

In [50]:
def get_batch(source, i, seq_length):
    """Gets a single batch from source data at position i"""
    seq_len = min(seq_length, len(source) - 1 - i)
    data = source[i : i + seq_len]
    # predict the sequences shifted by one word
    target = source[i + 1 : i + 1 + seq_len].view(-1)
    return data, target

In [51]:
corpus = Corpus('/scratch2/mrenaudin/colorlessgreenRNNs/english_data')

In [53]:
print(corpus.train.shape)
print(corpus.valid.shape)
print(corpus.test.shape)


torch.Size([83058298])
torch.Size([10391172])
torch.Size([10366477])


In [58]:
corpus2 = Corpus('/scratch2/mrenaudin/colorlessgreenRNNs/english_data')

In [59]:
corpus2.train = tokenize(corpus2.dictionary, os.path.join('/scratch2/mrenaudin/colorlessgreenRNNs/english_data', 'train.txt'), shuffle=True)

In [64]:
corpus2.train.shape
corpus2.valid.shape
corpus2.test.shape

torch.Size([10366477])

In [23]:
import sys
import os

sys.path.append('/scratch2/mrenaudin/colorlessgreenRNNs')

from src.language_models import model as m
import torch
from evaluation_notebooks.utils import BLiMPDataset, collate_fn_blimp
from src.language_models.dictionary_corpus import Dictionary
from torch.utils.data import DataLoader
from collections import defaultdict
from pathlib import Path
from tqdm import tqdm
import torch.nn.functional as F
import numpy as np


def compute_seq_nll(data, model, cache, nheads, temperature, gumbel):
    seq_len, batch_size = data.shape
    mask = (data!=0).float()

    output, cache = model(data, cache, nheads, temperature, gumbel)
    targets = data[1:, :]

    log_probs = F.log_softmax(output, dim=-1)
    log_probs = log_probs[:-1]
    log_probs = log_probs.swapaxes(0,1)

    nll_loss = F.nll_loss(
            log_probs.reshape(-1, log_probs.size(-1)),
            targets.reshape(-1),
            reduction='none'
        )
    
    nll_loss=nll_loss.reshape(seq_len - 1, batch_size)
    masked_nll_loss = nll_loss * mask[1:, :]
    sequence_nll = masked_nll_loss.sum(dim=1)
    return -sequence_nll
 
def eval_one_task(model, 
                  data_path, 
                  blimp_task, 
                  batch_size, 
                  gumbel, 
                  nheads, 
                  temperature, 
                  device
                  ):
    
    
    dictionary = Dictionary(data_path)
    blimp = BLiMPDataset(blimp_task, dictionary)
    test_dataloader = DataLoader(blimp, batch_size, collate_fn = collate_fn_blimp)
    
    correct_predictions = 0
    total_predictions = 0
    
    with torch.no_grad():
        for batch in test_dataloader:

            good = batch['encoded_good'].transpose(0,1).to(device)
            bad = batch['encoded_bad'].transpose(0,1).to(device)
            batch_size = good.size(0)
            
            cache_good = model.init_cache(good, 1)
            seq_nll_good = compute_seq_nll(good, model, cache_good, nheads, temperature, gumbel)
            cache_bad = model.init_cache(bad, 1)
            seq_nll_bad = compute_seq_nll(bad, model, cache_bad, nheads, temperature, gumbel)
            min_len = min(seq_nll_good.size(0), seq_nll_bad.size(0))
            seq_nll_good = seq_nll_good[:min_len]
            seq_nll_bad = seq_nll_bad[:min_len]
            predictions = (seq_nll_good > seq_nll_bad).cpu().numpy()
            
            correct_predictions += np.sum(predictions)
            total_predictions += batch_size
                
    accuracy = correct_predictions / total_predictions
    print(f"Accuracy on {test_dataloader.dataset.dataset.config_name}: {accuracy * 100:.2f}%")
    return accuracy

def eval_all_blimp(checkpoint, 
                  data_path, 
                  gumbel, 
                  nheads,
                  temperature,
                  hidden_dim, 
                  device):
    
    model = m.CBR_RNN(50001, hidden_dim, hidden_dim, nheads, 0.5, device)
    model=model.to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    checkpoint_results = {}
    
    blimp_tasks = ['adjunct_island', 
                   'anaphor_gender_agreement', 
                   'anaphor_number_agreement', 
                   'animate_subject_passive', 
                   'animate_subject_trans', 
                   'causative', 
                   'complex_NP_island', 
                   'coordinate_structure_constraint_complex_left_branch', 
                   'coordinate_structure_constraint_object_extraction', 
                   'determiner_noun_agreement_1', 
                   'determiner_noun_agreement_2', 
                   'determiner_noun_agreement_irregular_1', 
                   'determiner_noun_agreement_irregular_2', 
                   'determiner_noun_agreement_with_adj_2', 
                   'determiner_noun_agreement_with_adj_irregular_1', 
                   'determiner_noun_agreement_with_adj_irregular_2', 
                   'determiner_noun_agreement_with_adjective_1', 
                   'distractor_agreement_relational_noun', 
                   'distractor_agreement_relative_clause', 
                   'drop_argument', 'ellipsis_n_bar_1', 'ellipsis_n_bar_2', 
                   'existential_there_object_raising', 'existential_there_quantifiers_1', 
                   'existential_there_quantifiers_2', 'existential_there_subject_raising', 
                   'expletive_it_object_raising', 'inchoative', 'intransitive', 
                   'irregular_past_participle_adjectives', 'irregular_past_participle_verbs', 
                   'irregular_plural_subject_verb_agreement_1', 'irregular_plural_subject_verb_agreement_2', 
                   'left_branch_island_echo_question', 'left_branch_island_simple_question', 
                   'matrix_question_npi_licensor_present', 'npi_present_1', 'npi_present_2', 
                   'only_npi_licensor_present', 'only_npi_scope', 'passive_1', 
                   'passive_2', 'principle_A_c_command', 'principle_A_case_1', 
                   'principle_A_case_2', 'principle_A_domain_1', 'principle_A_domain_2', 
                   'principle_A_domain_3', 'principle_A_reconstruction', 
                   'regular_plural_subject_verb_agreement_1', 'regular_plural_subject_verb_agreement_2', 
                   'sentential_negation_npi_licensor_present', 'sentential_negation_npi_scope', 
                   'sentential_subject_island', 'superlative_quantifiers_1', 
                   'superlative_quantifiers_2', 'tough_vs_raising_1', 'tough_vs_raising_2', 
                   'transitive', 'wh_island', 'wh_questions_object_gap', 'wh_questions_subject_gap', 
                   'wh_questions_subject_gap_long_distance', 'wh_vs_that_no_gap', 'wh_vs_that_no_gap_long_distance', 
                   'wh_vs_that_with_gap', 'wh_vs_that_with_gap_long_distance']
    
    for task in blimp_tasks:
        checkpoint_results[task]= eval_one_task(model, 
                                                data_path, 
                                                task, 
                                                512, 
                                                gumbel, 
                                                nheads, 
                                                temperature,
                                                device
                                                )
    return checkpoint_results
        


In [24]:
checkpoint = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/wm_xp/experiments/tetetetetetest/results/h8_heads1_t0.1_true/h8_heads1_t0.1_gumbel/epoch_1.pt', map_location='cpu')

In [25]:
data_path = '/scratch2/mrenaudin/colorlessgreenRNNs/english_data'

In [26]:
results = eval_all_blimp(checkpoint, data_path, True, 1,0.1,8, 'cpu')

Accuracy on adjunct_island: 32.00%
Accuracy on anaphor_gender_agreement: 16.67%
Accuracy on anaphor_number_agreement: 25.00%
Accuracy on animate_subject_passive: 63.64%
Accuracy on animate_subject_trans: 77.78%
Accuracy on causative: 68.42%
Accuracy on complex_NP_island: 22.22%
Accuracy on coordinate_structure_constraint_complex_left_branch: 44.83%
Accuracy on coordinate_structure_constraint_object_extraction: 40.91%
Accuracy on determiner_noun_agreement_1: 36.84%
Accuracy on determiner_noun_agreement_2: 27.78%
Accuracy on determiner_noun_agreement_irregular_1: 58.82%
Accuracy on determiner_noun_agreement_irregular_2: 41.18%
Accuracy on determiner_noun_agreement_with_adj_2: 25.00%
Accuracy on determiner_noun_agreement_with_adj_irregular_1: 42.86%
Accuracy on determiner_noun_agreement_with_adj_irregular_2: 52.38%
Accuracy on determiner_noun_agreement_with_adjective_1: 30.43%
Accuracy on distractor_agreement_relational_noun: 27.27%
Accuracy on distractor_agreement_relative_clause: 17.65%

In [43]:
import sys
import os

sys.path.append('/scratch2/mrenaudin/colorlessgreenRNNs')

import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from src.language_models.dictionary_corpus import Dictionary
from collections import defaultdict
import torch.nn as nn
import src.language_models.model as m
import math
import torch.nn.functional as F
from wm_tests.utils import WMTestDataset, collate_fn, eval
from pathlib import Path
from tqdm import tqdm

def create_dataloader(base_path, file_suffix, batch_size, dictionary):

    data_file = f'{base_path}/{file_suffix}.txt'
    marker_file = f'{base_path}/{file_suffix}_markers.txt'
    dataset = WMTestDataset(data_file, marker_file, dictionary)
    return DataLoader(dataset, batch_size, collate_fn=collate_fn)

def load_test(cat_or_rand, sce, batch_size, dictionary, test_types):

    base_path = '/scratch2/mrenaudin/colorlessgreenRNNs/wm_tests/rnn_input_files'
        
    
    prefix_map = {
        'cat': f'categorized_lists_sce{sce}',
        'rand': f'random_lists_sce{sce}'
    }
    
    if cat_or_rand not in prefix_map:
        raise ValueError(f"cat_or_rand must be 'cat' or 'rand', got: {cat_or_rand}")
    
    prefix = prefix_map[cat_or_rand]
    
    dataloaders = {
        f'{cat_or_rand}_s{sce}_{test_type}_dataloader': create_dataloader(base_path, f'{prefix}_{test_type}',230, dictionary)
        for test_type in test_types
    }
    
    return dataloaders
         
def eval_repeat_surprisal(checkpoint, 
         data_path,
         cat_or_rand,
         sce,
         nheads,
         gumbel,
         hidden_dim,
         device,
         batch_size=230
         ):
    
    res = {}

    data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"
    dictionary = Dictionary(data_path)

    model = m.CBR_RNN(50001, hidden_dim, hidden_dim, nheads, 0.5, device)
    model=model.to(device)
    model.load_state_dict(checkpoint['model_state_dict'])

    test_types = ['control', 'permute', 'repeat']

    dataloaders = load_test(cat_or_rand, sce, batch_size, dictionary, test_types)
    temperature = checkpoint['temperature']
    print(dataloaders)
    for type in test_types:
        print(type)
        print(dataloaders.keys)
        dataloader = dataloaders[f'{cat_or_rand}_s{sce}_{type}_dataloader']
        res[type]= eval(model, dataloader, nheads, temperature, gumbel,device)

    return res 



In [44]:
res = eval_repeat_surprisal(checkpoint, 
         data_path,
         'cat',
         1,
         1,
         True,
         8,
         'cpu',
         batch_size=230
         )

{'cat_s1_control_dataloader': <torch.utils.data.dataloader.DataLoader object at 0x7fe0a41c8ca0>, 'cat_s1_permute_dataloader': <torch.utils.data.dataloader.DataLoader object at 0x7fe0245052b0>, 'cat_s1_repeat_dataloader': <torch.utils.data.dataloader.DataLoader object at 0x7fe0244dbd00>}
control
<built-in method keys of dict object at 0x7fe0190c0ac0>


KeyboardInterrupt: 

In [38]:
dataloaders

NameError: name 'dataloaders' is not defined